# 07 - Optuna with 5-Fold TimeSeries CV

## Goal
- optimize MLP hyperparameters with Optuna,
- evaluate each trial in 5-fold TimeSeriesSplit,
- log optimization progress to TensorBoard,
- save best trial and top-3 trial configs for notebook 08.


## Mapping to Lab6/7 Points
- **Punkt 7**: Optuna HPO w schemacie 5-fold `TimeSeriesSplit` (bez losowego seedowania split?w w objective), logowanie do TensorBoard, zapis podsumowa? i top-3 triali.


In [ ]:
import json
import os
from datetime import datetime
import numpy as np
import optuna
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import RobustScaler, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from stylin import InfoDisplayStyler

In [ ]:
styler = InfoDisplayStyler()

In [11]:
DATA_PATH = '../../data/raw/ethusdt_1h.csv'
CLEAN_PATH = '../../data/clean/'
LABELED_PATH = '../../data/labeled/'
PROCESSED_PATH = '../../data/processed/'
LOGS_PATH = '../../logs/'
REPORTS_PATH = '../../reports/'

TRAIN_SELECTED_FILE = PROCESSED_PATH + 'train_selected.csv'
VAL_SELECTED_FILE = PROCESSED_PATH + 'val_selected.csv'
SELECTED_FEATURES_FILE = PROCESSED_PATH + 'selected_feature_columns.csv'

assert os.path.exists(TRAIN_SELECTED_FILE), 'Missing train_selected.csv. Run notebook 04 first.'
assert os.path.exists(VAL_SELECTED_FILE), 'Missing val_selected.csv. Run notebook 04 first.'
assert os.path.exists(SELECTED_FEATURES_FILE), 'Missing selected_feature_columns.csv. Run notebook 04 first.'

os.makedirs(LOGS_PATH, exist_ok=True)
os.makedirs(LOGS_PATH + 'optuna/', exist_ok=True)
os.makedirs(LOGS_PATH + 'tensorboard/optuna/', exist_ok=True)
os.makedirs(REPORTS_PATH, exist_ok=True)

train_df = pd.read_csv(TRAIN_SELECTED_FILE)
val_df = pd.read_csv(VAL_SELECTED_FILE)
selected_features = pd.read_csv(SELECTED_FEATURES_FILE)['feature_column'].tolist()

for df in [train_df, val_df]:
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')

train_val_df = pd.concat([train_df, val_df], axis=0).sort_values('timestamp').reset_index(drop=True)

X_all = train_val_df[selected_features].to_numpy(dtype=np.float32)
y_all = train_val_df['target'].to_numpy(dtype=np.int64)

setup_report = pd.DataFrame({
    'metric': ['rows_train', 'rows_val', 'rows_train_val', 'num_features'],
    'value': [len(train_df), len(val_df), len(train_val_df), len(selected_features)],
})
styler.style_me(setup_report, title='Optuna input setup')


,metric,value
0,rows_train,32326
1,rows_val,6937
2,rows_train_val,39263
3,num_features,38


## 1. Model, dataset and training utilities


In [12]:
class TabularDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def build_activation(name: str) -> nn.Module:
    name = name.lower()
    if name == 'relu':
        return nn.ReLU()
    if name == 'gelu':
        return nn.GELU()
    if name == 'leaky_relu':
        return nn.LeakyReLU(negative_slope=0.1)
    raise ValueError(f'Unsupported activation: {name}')


class MLPClassifier(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dims: list[int],
        num_classes: int,
        activation: str,
        dropout: float,
        use_batchnorm: bool,
    ):
        super().__init__()

        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(build_activation(activation))
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def build_optimizer(name: str, model: nn.Module, lr: float, weight_decay: float):
    name = name.lower()
    if name == 'adam':
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    if name == 'adamw':
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    raise ValueError(f'Unsupported optimizer: {name}')


def get_scaler(scaler_type: str):
    scaler_type = scaler_type.lower()
    if scaler_type == 'standard':
        return StandardScaler()
    if scaler_type == 'robust':
        return RobustScaler()
    raise ValueError(f'Unsupported scaler_type: {scaler_type}')


def compute_class_weights(y: np.ndarray, num_classes: int) -> torch.Tensor:
    counts = np.bincount(y, minlength=num_classes)
    weights = len(y) / (num_classes * np.maximum(counts, 1))
    return torch.tensor(weights, dtype=torch.float32)


def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    clip_grad_norm: float | None = None,
) -> float:
    model.train()
    running_loss = 0.0
    n_batches = 0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()

        if clip_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad_norm)

        optimizer.step()
        running_loss += float(loss.item())
        n_batches += 1

    return running_loss / max(1, n_batches)


@torch.no_grad()
def validate_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> dict[str, float]:
    model.eval()
    running_loss = 0.0
    n_batches = 0
    all_preds = []
    all_targets = []

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        preds = torch.argmax(logits, dim=1)
        all_preds.append(preds.cpu().numpy())
        all_targets.append(y_batch.cpu().numpy())

        running_loss += float(loss.item())
        n_batches += 1

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_targets)

    return {
        'val_loss': running_loss / max(1, n_batches),
        'f1_macro': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
    }


## 2. Optuna config


In [13]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

N_SPLITS = 5
N_TRIALS = 30
EPOCHS_PER_FOLD = 20
EARLY_STOPPING_PATIENCE = 5

STUDY_NAME = 'eth_mlp_optimization'
STORAGE_URL = f"sqlite:///{os.path.abspath(LOGS_PATH + 'optuna/optuna.db').replace('\\', '/')}"
DIRECTION = 'maximize'

PRUNER = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)


## 3. Hyperparameter search space


In [14]:
def suggest_params(trial: optuna.Trial) -> dict:
    n_layers = trial.suggest_int('n_layers', 2, 5)
    hidden_dims = [trial.suggest_int(f'hidden_dim_{i+1}', 32, 512, step=32) for i in range(n_layers)]

    return {
        'hidden_dims': hidden_dims,
        'dropout': trial.suggest_float('dropout', 0.0, 0.5),
        'activation': trial.suggest_categorical('activation', ['relu', 'leaky_relu', 'gelu']),
        'use_batchnorm': trial.suggest_categorical('use_batchnorm', [True, False]),
        'optimizer': trial.suggest_categorical('optimizer', ['adam', 'adamw']),
        'lr': trial.suggest_float('lr', 1e-5, 1e-2, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-7, 1e-3, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [32, 64, 128, 256]),
        'scaler_type': trial.suggest_categorical('scaler_type', ['standard', 'robust']),
        'clip_grad_norm': trial.suggest_float('clip_grad_norm', 0.5, 5.0),
    }


## 4. Objective with 5-fold TimeSeriesSplit (Task point 7)


In [15]:
def objective(trial: optuna.Trial) -> float:
    params = suggest_params(trial)

    # Required by task: do not randomize fold splitting in objective.
    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    fold_scores = []

    trial_log_dir = LOGS_PATH + f'tensorboard/optuna/trial_{trial.number}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
    writer = SummaryWriter(log_dir=trial_log_dir)

    try:
        for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X_all), start=1):
            X_train_fold, y_train_fold = X_all[train_idx], y_all[train_idx]
            X_val_fold, y_val_fold = X_all[val_idx], y_all[val_idx]

            scaler = get_scaler(params['scaler_type'])
            X_train_scaled = scaler.fit_transform(X_train_fold).astype(np.float32)
            X_val_scaled = scaler.transform(X_val_fold).astype(np.float32)

            train_loader = DataLoader(
                TabularDataset(X_train_scaled, y_train_fold),
                batch_size=params['batch_size'],
                shuffle=True,
                num_workers=0,
            )
            val_loader = DataLoader(
                TabularDataset(X_val_scaled, y_val_fold),
                batch_size=params['batch_size'],
                shuffle=False,
                num_workers=0,
            )

            model = MLPClassifier(
                input_dim=X_train_scaled.shape[1],
                hidden_dims=params['hidden_dims'],
                num_classes=len(np.unique(y_all)),
                activation=params['activation'],
                dropout=params['dropout'],
                use_batchnorm=params['use_batchnorm'],
            ).to(DEVICE)

            class_weights = compute_class_weights(y_train_fold, num_classes=len(np.unique(y_all))).to(DEVICE)
            criterion = nn.CrossEntropyLoss(weight=class_weights)
            optimizer = build_optimizer(params['optimizer'], model, params['lr'], params['weight_decay'])

            best_fold_f1 = -np.inf
            bad_epochs = 0

            for epoch in range(1, EPOCHS_PER_FOLD + 1):
                train_loss = train_one_epoch(
                    model=model,
                    dataloader=train_loader,
                    optimizer=optimizer,
                    criterion=criterion,
                    device=DEVICE,
                    clip_grad_norm=params['clip_grad_norm'],
                )
                val_metrics = validate_one_epoch(
                    model=model,
                    dataloader=val_loader,
                    criterion=criterion,
                    device=DEVICE,
                )

                writer.add_scalar(f'trial_{trial.number}/fold_{fold_idx}/train_loss', train_loss, epoch)
                writer.add_scalar(f'trial_{trial.number}/fold_{fold_idx}/val_loss', val_metrics['val_loss'], epoch)
                writer.add_scalar(f'trial_{trial.number}/fold_{fold_idx}/f1_macro', val_metrics['f1_macro'], epoch)

                if val_metrics['f1_macro'] > best_fold_f1:
                    best_fold_f1 = val_metrics['f1_macro']
                    bad_epochs = 0
                else:
                    bad_epochs += 1
                    if bad_epochs >= EARLY_STOPPING_PATIENCE:
                        break

            fold_scores.append(float(best_fold_f1))

            mean_so_far = float(np.mean(fold_scores))
            trial.report(mean_so_far, step=fold_idx)
            writer.add_scalar(f'trial_{trial.number}/mean_f1_so_far', mean_so_far, fold_idx)

            if trial.should_prune():
                raise optuna.TrialPruned()

        final_score = float(np.mean(fold_scores))
        writer.add_scalar(f'trial_{trial.number}/final_mean_f1', final_score, 0)
        return final_score

    finally:
        writer.close()


## 5. Run optimization


In [16]:
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_URL,
    direction=DIRECTION,
    load_if_exists=True,
    pruner=PRUNER,
)

study.optimize(objective, n_trials=N_TRIALS)

styler.show_line(STUDY_NAME, title='Study name')
styler.show_line(STORAGE_URL, title='Storage')
styler.show_line(study.best_trial.number, title='Best trial number')
styler.show_line(study.best_value, title='Best mean CV f1_macro')
styler.style_me(pd.DataFrame([study.best_params]), title='Best params')


[I 2026-05-02 16:43:29,403] Using an existing study with name 'eth_mlp_optimization' instead of creating a new one.
[I 2026-05-02 16:48:55,506] Trial 5 finished with value: 0.39350144548090177 and parameters: {'n_layers': 5, 'hidden_dim_1': 384, 'hidden_dim_2': 512, 'hidden_dim_3': 192, 'hidden_dim_4': 160, 'hidden_dim_5': 448, 'dropout': 0.040456551864980095, 'activation': 'relu', 'use_batchnorm': False, 'optimizer': 'adamw', 'lr': 4.680077818280909e-05, 'weight_decay': 4.343712746093754e-06, 'batch_size': 32, 'scaler_type': 'standard', 'clip_grad_norm': 0.6276495024983375}. Best is trial 5 with value: 0.39350144548090177.
[I 2026-05-02 16:49:12,046] Trial 6 pruned. 
[I 2026-05-02 16:50:53,670] Trial 7 pruned. 
[I 2026-05-02 16:51:47,662] Trial 8 pruned. 
[I 2026-05-02 16:52:03,296] Trial 9 pruned. 
[I 2026-05-02 16:52:34,702] Trial 10 pruned. 
[I 2026-05-02 16:54:38,366] Trial 11 pruned. 
[I 2026-05-02 16:55:55,407] Trial 12 pruned. 
[I 2026-05-02 16:57:15,265] Trial 13 pruned. 
[I 2

,n_layers,hidden_dim_1,hidden_dim_2,hidden_dim_3,hidden_dim_4,hidden_dim_5,dropout,activation,use_batchnorm,optimizer,lr,weight_decay,batch_size,scaler_type,clip_grad_norm
0,5,128,384,128,320,320,0.213133,relu,False,adam,0.000755,0.000015,64,standard,1.828619


## 6. Save summary and top-3 trials


In [17]:
trials_df = study.trials_dataframe(attrs=('number', 'value', 'state', 'params'))
trials_df.to_csv(REPORTS_PATH + 'optuna_trials.csv', index=False)

completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
completed_sorted = sorted(completed, key=lambda t: t.value, reverse=True)
top3 = completed_sorted[:3]

top3_rows = []
for rank, t in enumerate(top3, start=1):
    top3_rows.append({'rank': rank, 'trial_number': t.number, 'value': float(t.value), **t.params})

top3_df = pd.DataFrame(top3_rows)
top3_df.to_csv(REPORTS_PATH + 'optuna_top3.csv', index=False)

summary = {
    'study_name': STUDY_NAME,
    'best_trial_number': int(study.best_trial.number),
    'best_value': float(study.best_value),
    'best_params': study.best_params,
    'n_trials_total': int(len(study.trials)),
    'n_trials_complete': int(len(completed)),
    'top3': [
        {'rank': i + 1, 'trial_number': int(t.number), 'value': float(t.value), 'params': t.params}
        for i, t in enumerate(top3)
    ],
}

with open(REPORTS_PATH + 'optuna_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

styler.style_me(top3_df, title='Top-3 trials for notebook 08')


,rank,trial_number,value,n_layers,hidden_dim_1,hidden_dim_2,hidden_dim_3,hidden_dim_4,hidden_dim_5,dropout,activation,use_batchnorm,optimizer,lr,weight_decay,batch_size,scaler_type,clip_grad_norm
0,1,21,0.396377,5,128,384,128,320.000000,320.000000,0.213133,relu,False,adam,0.000755,0.000015,64,standard,1.828619
1,2,15,0.394122,3,352,480,128,nan,nan,0.257373,leaky_relu,False,adam,0.000161,0.000007,32,standard,3.822142
2,3,30,0.394113,5,256,224,160,288.000000,512.000000,0.122908,leaky_relu,False,adam,0.000778,0.000001,32,standard,1.588607


## 7. Optional Optuna visual diagnostics


In [18]:
try:
    import optuna.visualization as vis

    fig1 = vis.plot_optimization_history(study)
    fig1.show()

    fig2 = vis.plot_param_importances(study)
    fig2.show()
except Exception as e:
    print('Optional visualization skipped:', e)


Optional visualization skipped: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.
